# 01 - Collaborative Filtering: ALS with `implicit`
**Thesis Project**: Deep Hybrid Recommender System  
**Dataset**: Amazon Reviews (~45.6M positive interactions, 22M users × 3.7M items)  
**Optimal Rubric Config**: Factors=64, Iterations=20, Regularization=0.05, Confidence Scale=15.0


In [ ]:
# 1. Install dependencies
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "implicit>=0.7", "pyarrow", "scipy", "tqdm"
])
print("Implicit installed successfully.")


In [ ]:
import os, sys, time, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from tqdm import tqdm

print("Python:", sys.version)

def find_base_dir() -> Path:
    candidates = [
        Path("/kaggle/input/lastone/amazon_recs_output"),
        Path("/kaggle/input/datasets/nguynteo/lastone/amazon_recs_output"),
        Path("/kaggle/input/lastone"),
        Path("data"),
    ]
    for c in candidates:
        if (c / "train").exists() or (c / "test").exists():
            print(f"Using base dir: {c}")
            return c
    for p in Path("/kaggle/input").rglob("train"):
        if p.is_dir():
            print(f"Discovered train at: {p.parent}")
            return p.parent
    return Path("data")

BASE_DIR = find_base_dir()
OUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "artifacts/factors")
OUT_DIR.mkdir(parents=True, exist_ok=True)
NUM_USERS = 22_048_170
NUM_ITEMS = 3_686_940


In [ ]:
# 2. Load Train Interactions (rating >= 4.0)
train_files = sorted(BASE_DIR.glob("train/*.parquet"))
print(f"Found {len(train_files)} train parquet files")

dfs = []
for f in tqdm(train_files, desc="Loading train"):
    t = pq.read_table(f, columns=["user_idx", "item_idx", "rating"])
    d = t.to_pandas()
    d = d[d["rating"] >= 4.0]
    dfs.append(d[["user_idx", "item_idx"]])

train_df = pd.concat(dfs, ignore_index=True)
print(f"Total positive train interactions: {len(train_df):,}")

# Filter active subset of users to guarantee fast convergence & fit 30GB RAM
top_users = train_df["user_idx"].value_counts().nlargest(300000).index
active_df = train_df[train_df["user_idx"].isin(top_users)].copy()
del train_df, dfs

u_raw = active_df["user_idx"].to_numpy(dtype=np.int64)
i_raw = active_df["item_idx"].to_numpy(dtype=np.int64)
u_uniq, u_inv = np.unique(u_raw, return_inverse=True)
i_uniq, i_inv = np.unique(i_raw, return_inverse=True)

print(f"Compact Matrix: {len(u_uniq):,} active users x {len(i_uniq):,} active items ({len(active_df):,} interactions)")

# Implicit confidence scaling: c = 1.0 + alpha * r (standard alpha=15.0 for implicit feedback)
data_conf = np.full(len(u_inv), 15.0, dtype=np.float32)
user_item_matrix = csr_matrix((data_conf, (u_inv, i_inv)), shape=(len(u_uniq), len(i_uniq)))
print(f"Matrix built successfully. Non-zero entries: {user_item_matrix.nnz:,}")


In [ ]:
# 3. Train ALS with Optimal Rubric Hyperparameters
print("Starting ALS Training (factors=64, iters=20, reg=0.05, threads=all)...")
t0 = time.time()
als_model = AlternatingLeastSquares(
    factors=64,
    regularization=0.05,
    iterations=20,
    calculate_training_loss=True,
    num_threads=0,
    random_state=42
)
als_model.fit(user_item_matrix)
als_time = time.time() - t0
print(f"ALS fit completed in {als_time:.1f}s")

user_factors = np.asarray(als_model.user_factors, dtype=np.float32)
item_factors = np.asarray(als_model.item_factors, dtype=np.float32)
print(f"user_factors shape: {user_factors.shape}, item_factors shape: {item_factors.shape}")
del user_item_matrix, als_model


In [ ]:
# 4. Export Factors (Float16 + Float32 + ID mappings)
print("Saving ALS factor files...")
np.save(OUT_DIR / "als_user_factors_f16.npy", user_factors.astype(np.float16))
np.save(OUT_DIR / "als_item_factors_f16.npy", item_factors.astype(np.float16))
np.save(OUT_DIR / "als_user_factors_f32.npy", user_factors)
np.save(OUT_DIR / "als_item_factors_f32.npy", item_factors)
np.save(OUT_DIR / "als_user_ids.npy", u_uniq)
np.save(OUT_DIR / "als_item_ids.npy", i_uniq)

meta = {
    "model": "ALS",
    "library": "implicit",
    "factors": 64,
    "iterations": 20,
    "regularization": 0.05,
    "train_time_sec": als_time,
    "user_factors_shape": list(user_factors.shape),
    "item_factors_shape": list(item_factors.shape),
}
(OUT_DIR / "als_meta.json").write_text(json.dumps(meta, indent=2))
print("Factors and metadata saved successfully!")


In [ ]:
# 5. Evaluate on Test Set (HR@5/10/20, NDCG@5/10/20, MRR)
test_files = sorted(BASE_DIR.glob("test/*.parquet"))
dfs_test = []
for f in tqdm(test_files, desc="Loading test"):
    t = pq.read_table(f, columns=["user_idx", "item_idx", "rating"])
    d = t.to_pandas()
    d = d[d["rating"] >= 4.0]
    dfs_test.append(d[["user_idx", "item_idx"]])

test_df = pd.concat(dfs_test, ignore_index=True)

u_to_idx = {u: idx for idx, u in enumerate(u_uniq)}
i_to_idx = {it: idx for idx, it in enumerate(i_uniq)}

test_df["mapped_u"] = test_df["user_idx"].map(u_to_idx)
test_df["mapped_i"] = test_df["item_idx"].map(i_to_idx)
test_clean = test_df.dropna(subset=["mapped_u", "mapped_i"]).copy()
test_clean["mapped_u"] = test_clean["mapped_u"].astype(np.int64)
test_clean["mapped_i"] = test_clean["mapped_i"].astype(np.int64)

user_gt = test_clean.groupby("mapped_u")["mapped_i"].apply(set).to_dict()

rng = np.random.default_rng(42)
test_users_mapped = list(user_gt.keys())
eval_users = rng.choice(test_users_mapped, size=min(5000, len(test_users_mapped)), replace=False)
all_items_pool = np.arange(len(item_factors))

metrics = {
    "HR@5": 0.0, "HR@10": 0.0, "HR@20": 0.0,
    "NDCG@5": 0.0, "NDCG@10": 0.0, "NDCG@20": 0.0,
    "MRR": 0.0
}
cnt = 0
for u in tqdm(eval_users, desc="ALS Test Eval"):
    gts = list(user_gt[u])
    if not gts: continue
    gt = gts[0]
    
    negs = rng.choice(all_items_pool, size=150, replace=False)
    negs = [it for it in negs if it not in user_gt[u]][:99]
    cands = np.array([gt] + negs, dtype=np.int64)
    
    scores = item_factors[cands] @ user_factors[u]
    ranked = cands[np.argsort(-scores)]
    pos_list = np.where(ranked == gt)[0]
    if len(pos_list) == 0: continue
    pos = int(pos_list[0])
    
    if pos < 5: metrics["HR@5"] += 1.0
    if pos < 10: metrics["HR@10"] += 1.0
    if pos < 20: metrics["HR@20"] += 1.0
    if pos < 5: metrics["NDCG@5"] += 1.0 / math.log2(pos + 2)
    if pos < 10: metrics["NDCG@10"] += 1.0 / math.log2(pos + 2)
    if pos < 20: metrics["NDCG@20"] += 1.0 / math.log2(pos + 2)
    metrics["MRR"] += 1.0 / (pos + 1)
    cnt += 1

if cnt > 0:
    for k in metrics: metrics[k] /= cnt

print("\n" + "="*50)
print(f"ALS EVALUATION RESULTS (N={cnt} users):")
for k, v in metrics.items():
    print(f"  {k:8s}: {v:.4f}")
print("="*50)

eval_res = {"model": "ALS (implicit)", "metrics": metrics, "n_eval_users": cnt, "train_time_sec": als_time}
(OUT_DIR / "als_evaluation_metrics.json").write_text(json.dumps(eval_res, indent=2))
print("Finished ALS notebook successfully!")
